# Sampling primitives without a language model

Synthetic logits isolate the mathematics. Temperature precedes Top-K/Top-P; the final probabilities are normalized after masking. Greedy entropy zero describes a deterministic policy, not factual confidence.

In [ ]:
import os
from pathlib import Path
import uuid
project = Path.cwd()
if not (project/'configs').is_dir(): project = project.parent
assert (project/'configs').is_dir(), 'Launch from the project or notebooks directory'
os.chdir(project)
run = project/'artifacts'/('notebook-'+uuid.uuid4().hex[:12])

In [ ]:
from llm_decoding.reports import demo
record = demo(run)
from IPython.display import display
from PIL import Image
display(Image.open(run/'distributions.png'))
for policy in record['distributions']:
    print(policy['name'],policy['eligible_tokens'],policy['entropy_nats'])

In [ ]:
import torch
from llm_decoding.sampling import SamplingConfig,distribution,top_p_logits
uniform = top_p_logits(torch.zeros(4),.5)
assert torch.isfinite(uniform).tolist() == [True,True,False,False]
logits = torch.tensor([2.,1.,0.])
warm = distribution(logits,SamplingConfig(temperature=1.,top_k=0,top_p=.8))[1]
cold = distribution(logits,SamplingConfig(temperature=.25,top_k=0,top_p=.8))[1]
assert int(torch.isfinite(warm).sum()) == 2
assert int(torch.isfinite(cold).sum()) == 1
print('Temperature changes the nucleus, so the operation order matters.')

Exact-CDF and tied-logit membership follows the documented stable minimal-prefix contract. Top-K cutoff ties may retain more than K tokens. T=0 explicitly means argmax; negative temperatures fail. See ../docs/sampling.md for numerical and reference-test scope.